# Resultados da ablação — sMCI vs pMCI

Pipeline de visualização para apresentação e artigo.

**Entrada:** `ablation_summary_*.csv` e `ablation_results_*.csv` em `csvs/abordagem_4_sMCI_pMCI_extremos/`

**Saída:** figuras PNG (300 dpi) e PDF em `csvs/abordagem_4_sMCI_pMCI_extremos/figures/`

**Protocolo:** nested CV 5×5, seleção MRMR por bloco (corr + var + MRMR), modelos SVM / RF / MLP.

# Filters

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch

# MODALITY = "vol", "shape", "texture", "disp", "all"
MODALITY = "vol"  # alinhar com 3_ablation.ipynb 
RESULTS_DIR = Path(f"csvs/longitudinal_4_groups/ablation_results/{MODALITY}")
RESULTS_PATH = RESULTS_DIR / "ablation_results_all.csv"
SUMMARY_PATH = RESULTS_DIR / "ablation_summary.csv"


MODEL_ORDER = ["svm", "rf", "mlp"]
MOD_ORDER = ["vol", "shape", "texture", "disp", "all"]
COMBAT_ORDER = ["sem ComBat", "ComBat"]
COLOR_MAP = {"svm": "#4477AA", "rf": "#EE6677", "mlp": "#228833"}


def _coerce_with_combat(series: pd.Series) -> pd.Series:
    if series.dtype == bool:
        return series
    return series.astype(str).str.strip().str.lower().isin(("true", "1", "yes"))


def _load_summary_if_needed() -> pd.DataFrame:
    if "summary" in globals() and globals().get("summary") is not None:
        return globals()["summary"]
    if "SUMMARY_PATH" not in globals():
        raise NameError("Rode a célula de config (imports + RESULTS_DIR).")
    summary_path = globals()["SUMMARY_PATH"]
    if not Path(summary_path).is_file():
        raise FileNotFoundError(f"summary ausente: {summary_path}")
    return pd.read_csv(summary_path)


summary = _load_summary_if_needed()
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

plot_df = summary.copy()
if "with_combat" not in plot_df.columns:
    raise KeyError("summary sem coluna with_combat")
plot_df["with_combat"] = _coerce_with_combat(plot_df["with_combat"])
plot_df["combat_label"] = plot_df["with_combat"].map({True: "ComBat", False: "sem ComBat"})
if plot_df["combat_label"].isna().any():
    raise ValueError("with_combat inválido — valores não mapeiam para ComBat/sem ComBat")

plot_df["model_key"] = pd.Categorical(
    plot_df["model_key"].astype(str), categories=MODEL_ORDER, ordered=True
)
plot_df["modality"] = pd.Categorical(
    plot_df["modality"].astype(str), categories=MOD_ORDER, ordered=True
)

tasks = sorted(plot_df["task"].astype(str).unique())
combat_labels = [c for c in COMBAT_ORDER if c in set(plot_df["combat_label"])]
if not tasks or not combat_labels:
    raise ValueError(f"Nada para plotar: tasks={tasks}, combat={combat_labels}")

# 1) Heatmap AUC: modalidade × modelo (painel por task × ComBat)
fig, axes = plt.subplots(
    len(tasks), len(combat_labels),
    figsize=(4 * len(combat_labels), 3 * len(tasks)),
    squeeze=False,
)
for i, task_id in enumerate(tasks):
    for j, combat in enumerate(combat_labels):
        ax = axes[i, j]
        sub = plot_df[(plot_df["task"].astype(str) == task_id) & (plot_df["combat_label"] == combat)]
        if sub.empty:
            ax.axis("off")
            continue
        pivot = sub.pivot_table(
            index="model_key", columns="modality", values="auc_pooled", aggfunc="first", observed=False
        )
        pivot = pivot.reindex(index=MODEL_ORDER, columns=MOD_ORDER)
        sns.heatmap(
            pivot, annot=True, fmt=".2f", cmap="YlOrRd", vmin=0.5, vmax=1.0,
            ax=ax, cbar=(i == 0 and j == len(combat_labels) - 1),
            cbar_kws={"label": "AUC"} if (i == 0 and j == len(combat_labels) - 1) else None,
        )
        ax.set_title(f"{task_id}\n{combat}", fontsize=9)
        ax.set_xlabel("")
        ax.set_ylabel("modelo" if j == 0 else "")
fig.suptitle("AUC pooled (predições externas agregadas)", fontsize=12)
plt.tight_layout()
out_heat = RESULTS_DIR / "ablation_summary_heatmap_auc.png"
# fig.savefig(out_heat, dpi=150, bbox_inches="tight")
plt.show()
print(f"Figura: {out_heat}")

# 2) AUC com barras + std (um gráfico por task)
for task_id in tasks:
    sub = plot_df[plot_df["task"].astype(str) == task_id]
    fig, axes = plt.subplots(1, len(combat_labels), figsize=(5 * len(combat_labels), 4), squeeze=False)
    for ax, combat in zip(axes.flat, combat_labels):
        part = sub[sub["combat_label"] == combat]
        labels, means, stds, colors = [], [], [], []
        for mod in MOD_ORDER:
            for model in MODEL_ORDER:
                row = part[(part["modality"].astype(str) == mod) & (part["model_key"].astype(str) == model)]
                if row.empty:
                    continue
                r = row.iloc[0]
                labels.append(f"{mod}\n{model}")
                means.append(float(r["auc_mean"]))
                stds.append(float(r["auc_std"]))
                colors.append(COLOR_MAP[model])
        if labels:
            ax.bar(labels, means, yerr=stds, capsize=3, color=colors, ecolor="gray")
        else:
            ax.text(0.5, 0.5, "sem dados", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(combat)
        ax.set_ylabel("AUC")
        ax.tick_params(axis="x", labelsize=7)
        ax.axhline(0.5, color="gray", ls="--", lw=0.8)
        ax.set_ylim(0.4, 1.0)
    fig.suptitle(f"Task {task_id} — AUC ± std", fontsize=11)
    plt.tight_layout()
    out = RESULTS_DIR / f"ablation_summary_bars_{task_id}.png"
    # fig.savefig(out, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Figura: {out}")

# 3) Melhor config por task × modalidade
best = (
    plot_df.sort_values("auc_mean", ascending=False)
    .groupby(["task", "modality", "combat_label"], as_index=False, observed=True)
    .first()
)
fig, ax = plt.subplots(figsize=(9, max(4, 0.32 * len(best))))
y_labels = (
    best["task"].astype(str) + " | " + best["modality"].astype(str) + " | " + best["combat_label"]
)
ax.barh(
    range(len(best)),
    best["auc_mean"],
    xerr=best["auc_std"],
    color=best["model_key"].astype(str).map(COLOR_MAP),
    capsize=3,
    ecolor="gray",
)
ax.set_yticks(range(len(best)))
ax.set_yticklabels(y_labels, fontsize=8)
ax.set_xlabel("AUC médio ± std")
ax.axvline(0.5, color="gray", ls="--", lw=0.8)
ax.set_title("Melhor modelo (maior AUC) por task × modalidade")
ax.legend(handles=[Patch(facecolor=c, label=k.upper()) for k, c in COLOR_MAP.items()])
plt.tight_layout()
out_best = RESULTS_DIR / "ablation_summary_best_auc.png"
# fig.savefig(out_best, dpi=150, bbox_inches="tight")
plt.show()
print(f"Figura: {out_best}")


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch

# MODALITY = "vol", "shape", "texture", "disp", "all"
MODALITY = "vol"  # alinhar com 3_ablation.ipynb 
RESULTS_DIR = Path(f"csvs/longitudinal_4_groups/ablation_results/{MODALITY}")
RESULTS_PATH = RESULTS_DIR / "ablation_results_all_raw.csv"
SUMMARY_PATH = RESULTS_DIR / "ablation_summary_raw.csv"


MODEL_ORDER = ["svm", "rf", "mlp"]
MOD_ORDER = ["vol", "shape", "texture", "disp", "all"]
COMBAT_ORDER = ["sem ComBat", "ComBat"]
COLOR_MAP = {"svm": "#4477AA", "rf": "#EE6677", "mlp": "#228833"}


def _coerce_with_combat(series: pd.Series) -> pd.Series:
    if series.dtype == bool:
        return series
    return series.astype(str).str.strip().str.lower().isin(("true", "1", "yes"))


def _load_summary_if_needed() -> pd.DataFrame:
    if "summary" in globals() and globals().get("summary") is not None:
        return globals()["summary"]
    if "SUMMARY_PATH" not in globals():
        raise NameError("Rode a célula de config (imports + RESULTS_DIR).")
    summary_path = globals()["SUMMARY_PATH"]
    if not Path(summary_path).is_file():
        raise FileNotFoundError(f"summary ausente: {summary_path}")
    return pd.read_csv(summary_path)


summary = _load_summary_if_needed()
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

plot_df = summary.copy()
if "with_combat" not in plot_df.columns:
    raise KeyError("summary sem coluna with_combat")
plot_df["with_combat"] = _coerce_with_combat(plot_df["with_combat"])
plot_df["combat_label"] = plot_df["with_combat"].map({True: "ComBat", False: "sem ComBat"})
if plot_df["combat_label"].isna().any():
    raise ValueError("with_combat inválido — valores não mapeiam para ComBat/sem ComBat")

plot_df["model_key"] = pd.Categorical(
    plot_df["model_key"].astype(str), categories=MODEL_ORDER, ordered=True
)
plot_df["modality"] = pd.Categorical(
    plot_df["modality"].astype(str), categories=MOD_ORDER, ordered=True
)

tasks = sorted(plot_df["task"].astype(str).unique())
combat_labels = [c for c in COMBAT_ORDER if c in set(plot_df["combat_label"])]
if not tasks or not combat_labels:
    raise ValueError(f"Nada para plotar: tasks={tasks}, combat={combat_labels}")

# 1) Heatmap AUC: modalidade × modelo (painel por task × ComBat)
fig, axes = plt.subplots(
    len(tasks), len(combat_labels),
    figsize=(4 * len(combat_labels), 3 * len(tasks)),
    squeeze=False,
)
for i, task_id in enumerate(tasks):
    for j, combat in enumerate(combat_labels):
        ax = axes[i, j]
        sub = plot_df[(plot_df["task"].astype(str) == task_id) & (plot_df["combat_label"] == combat)]
        if sub.empty:
            ax.axis("off")
            continue
        pivot = sub.pivot_table(
            index="model_key", columns="modality", values="auc_pooled", aggfunc="first", observed=False
        )
        pivot = pivot.reindex(index=MODEL_ORDER, columns=MOD_ORDER)
        sns.heatmap(
            pivot, annot=True, fmt=".2f", cmap="YlOrRd", vmin=0.5, vmax=1.0,
            ax=ax, cbar=(i == 0 and j == len(combat_labels) - 1),
            cbar_kws={"label": "AUC"} if (i == 0 and j == len(combat_labels) - 1) else None,
        )
        ax.set_title(f"{task_id}\n{combat}", fontsize=9)
        ax.set_xlabel("")
        ax.set_ylabel("modelo" if j == 0 else "")
fig.suptitle("AUC pooled (predições externas agregadas)", fontsize=12)
plt.tight_layout()
out_heat = RESULTS_DIR / "ablation_summary_heatmap_auc.png"
# fig.savefig(out_heat, dpi=150, bbox_inches="tight")
plt.show()
print(f"Figura: {out_heat}")

# 2) AUC com barras + std (um gráfico por task)
for task_id in tasks:
    sub = plot_df[plot_df["task"].astype(str) == task_id]
    fig, axes = plt.subplots(1, len(combat_labels), figsize=(5 * len(combat_labels), 4), squeeze=False)
    for ax, combat in zip(axes.flat, combat_labels):
        part = sub[sub["combat_label"] == combat]
        labels, means, stds, colors = [], [], [], []
        for mod in MOD_ORDER:
            for model in MODEL_ORDER:
                row = part[(part["modality"].astype(str) == mod) & (part["model_key"].astype(str) == model)]
                if row.empty:
                    continue
                r = row.iloc[0]
                labels.append(f"{mod}\n{model}")
                means.append(float(r["auc_mean"]))
                stds.append(float(r["auc_std"]))
                colors.append(COLOR_MAP[model])
        if labels:
            ax.bar(labels, means, yerr=stds, capsize=3, color=colors, ecolor="gray")
        else:
            ax.text(0.5, 0.5, "sem dados", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(combat)
        ax.set_ylabel("AUC")
        ax.tick_params(axis="x", labelsize=7)
        ax.axhline(0.5, color="gray", ls="--", lw=0.8)
        ax.set_ylim(0.4, 1.0)
    fig.suptitle(f"Task {task_id} — AUC ± std", fontsize=11)
    plt.tight_layout()
    out = RESULTS_DIR / f"ablation_summary_bars_{task_id}.png"
    # fig.savefig(out, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Figura: {out}")

# 3) Melhor config por task × modalidade
best = (
    plot_df.sort_values("auc_mean", ascending=False)
    .groupby(["task", "modality", "combat_label"], as_index=False, observed=True)
    .first()
)
fig, ax = plt.subplots(figsize=(9, max(4, 0.32 * len(best))))
y_labels = (
    best["task"].astype(str) + " | " + best["modality"].astype(str) + " | " + best["combat_label"]
)
ax.barh(
    range(len(best)),
    best["auc_mean"],
    xerr=best["auc_std"],
    color=best["model_key"].astype(str).map(COLOR_MAP),
    capsize=3,
    ecolor="gray",
)
ax.set_yticks(range(len(best)))
ax.set_yticklabels(y_labels, fontsize=8)
ax.set_xlabel("AUC médio ± std")
ax.axvline(0.5, color="gray", ls="--", lw=0.8)
ax.set_title("Melhor modelo (maior AUC) por task × modalidade")
ax.legend(handles=[Patch(facecolor=c, label=k.upper()) for k, c in COLOR_MAP.items()])
plt.tight_layout()
out_best = RESULTS_DIR / "ablation_summary_best_auc.png"
# fig.savefig(out_best, dpi=150, bbox_inches="tight")
plt.show()
print(f"Figura: {out_best}")


In [ ]:
# Estabilidade de atributos — frequência 0–100% por configuração isolada
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from ablation_analysis import (
    feature_freq_table,
    filter_ablation_config,
    plot_feature_stability,
    prepare_ablation_df,
)

MODALITY #= "vol"  # alinhar com célula anterior
RESULTS_DIR = Path(f"csvs/longitudinal_4_groups/ablation_results/{MODALITY}")
RESULTS_PATH = RESULTS_DIR / "ablation_results_all_raw.csv"

PLOT_ALL_CONFIGS = False
PLOT_TASK = "smci_pmci" # "CN_AD", "sMCI_pMCI", "CN_sMCI", "sMCI_AD", "CN_pMCI", "pMCI_AD"
PLOT_MODEL = "svm" # "svm", "rf", "mlp"
PLOT_COMBAT = True # False, True
PLOT_SELECTION = "raw" # "raw", "filters", "mrmr"
MODELS = ("svm", "rf", "mlp")
WITH_COMBAT_FLAGS = (False, True)

df_ablation = prepare_ablation_df(pd.read_csv(RESULTS_PATH))

if PLOT_ALL_CONFIGS:
    plot_specs = [
        (str(t), MODALITY, str(mk), bool(wc))
        for t in df_ablation["task"].astype(str).unique()
        for mk in MODELS
        for wc in WITH_COMBAT_FLAGS
    ]
else:
    plot_specs = [(PLOT_TASK, MODALITY, PLOT_MODEL, PLOT_COMBAT)]

for task_id, mod_id, model_key, with_combat in plot_specs:
    try:
        df_cfg = filter_ablation_config(
            df_ablation,
            task=task_id,
            modality=mod_id,
            model_key=model_key,
            with_combat=with_combat,
            selection_mode=PLOT_SELECTION,
        )
    except ValueError as exc:
        print(f"Pulando {task_id}/{mod_id}/{model_key}/combat={with_combat}: {exc}")
        continue

    freq = feature_freq_table(df_cfg, top_n=None)
    tag = f"{task_id}_{mod_id}_{model_key}_{'combat' if with_combat else 'nocombat'}"
    # freq_path = RESULTS_DIR / f"feature_freq_{tag}.csv"
    # freq.to_csv(freq_path, index=False)

    fig = plot_feature_stability(df_cfg, out_path=RESULTS_DIR / f"feature_stability_{tag}.png")
    plt.show()
    # print(f"Salvo: {freq_path} ({len(freq)} atributos)")
    plt.close(fig)
